# Development of State Estimators

In [ ]:
import numpy as np
import scipy
import casadi as cas

## Linear Kalman filter equations

In [ ]:
def linear_model_state_transition(A, B, xk, uk):
    xkp1 = (A @ xk + B @ uk)
    return xkp1


def linear_model_output(C, D, xk, uk):
    yk = (C @ xk + D @ uk)
    return yk


def kalman_update_f(C, R, xk_pred, Pk_pred, yk_pred, yk):
    """Compute the updated correction gain, state estimate, state error
    covariance, output estimate, and output error covariance in the
    current time instant of a Kalman filter (filtering form) given
    a new measurement.

    y_est(k|k-1) = C(k) * x_est(k|k-1) + D(k) * u(k)
    S(k) = C(k) * P(k|k-1) * C'(k) + R(k)
    Kf(k) = P(k|k-1) * C'(k) * S(k)^-1
    x_est(k|k) = x_est(k|k-1) + Kf * (y(k) - y_est(k|k-1))
    P(k|k) = P(k|k-1) - Kf(k) * S(k) * Kf'(k)
    y_est(k|k) = C(k) * x_est(k|k) + D(k) * u(k)

    Arguments:
    C : (ny, n) matrix
        Measurement matrix.
    R : (ny, ny) matrix
        Measurement error covariance.
    xk_pred : (n, 1) vector
        Prior estimate of the states, x_est(k|k-1).
    Pk_pred : (n, n) matrix
        Prior estimate of the state error covariance, P(k|k-1).
    yk_pred : (ny, 1) vector
        Prior estimate of the system output, y_est(k|k-1).
    yk : (ny, 1) vector
        System output measurement at time k.

    Returns:
    xk_est : (n, 1) vector
        Posterior estimate of the states, x_est(k|k).
    Pk : (n, n) matrix
        Covariance of state estimation error, P(k|k).
    Kf : (n, ny) matrix
        Correction gain matrix.
    Sk : (ny, ny) matrix
        Covariance of output prediction error, S(k).
    """

    # Error covariance of output prediction error
    Sk = C @ Pk_pred @ C.T + R

    # Correction gain
    # Kf = Pk_pred * C.T / Sk  # in MATLAB
    # Numpy equivalent
    # Kf = np.linalg.lstsq(Sk.T, (Pk_pred @ C.T).T, rcond=-1)[0].T
    Kf = cas.mrdivide(Pk_pred @ C.T, Sk)

    # Update state predictions (prior estimates) using
    # measurements from current time to produce 'a posteriori'
    # state estimates
    xk_est = xk_pred + Kf @ (yk - yk_pred)

    # Update error covariance of state estimates
    # Pk = Pk_pred - Kf * Sk * Kf';  # inaccurate
    # Pk = Pk_pred - Kf * C * Pk_pred

    # The following is the 'Joseph Stabilized' version of
    # the update equation which guarantees that Pk is positive
    # semi-definite in the presence of roundoff error.
    # (See p73 of Lewis etal. Book Optimal and Robust Estimation).
    z = cas.DM.eye(Pk_pred.shape[0]) - Kf @ C
    Pk = z @ Pk_pred @ z.T + Kf @ R @ Kf.T

    # Trick to ensure that Pk is always symmetric, in case of
    # rounding errors
    Pk = (Pk + Pk.T) / 2

    return xk_est, Pk, Kf, Sk


def kalman_predict_f(A, B, Q, xk_est, Pk, uk):
    """compute the predicted states, outputs, and state error
    covariance of a Kalman filter (filtering form) in the next
    time instant given the model parameters, the current error
    covariance, and known input.

    x_est(k+1|k) = A(k) * x_est(k|k) + B(k) * u(k)
    P(k+1|k) = A(k) * P(k|k) * A'(k) + Q(k)

    Arguments:
    A, B : matrices
        Discrete-time system model matrices (assumes D = 0).
    Q : matrix, size (n, n)
        State error covariance.
    xk_est : (n, 1) vector
        Posterior estimate of the states, x_est(k|k).
    Pk : (n, n) matrix
        State error covariance, P(k|k).
    uk : (nu, 1) vector
        Known system inputs at time k.

    Returns:
    xkp1_est : (n, 1) vector
        Predicted states at next time instant, x_est(k+1|k).
    Pk : (n, n) matrix
        Predicted state error covariance at next time instant,
        P(k+1|k).
    """

    # Make predictions of states and outputs in next time step
    # x_est(k+1|k) and y_est(k+1|k) i.e. based on the data up
    # to time k. These will be used as priors in the next time
    # step.
    xkp1_est = linear_model_state_transition(A, B, xk_est, uk)

    # Error covariance of predicted state estimates
    Pkp1 = A @ Pk @ A.T + Q

    return xkp1_est, Pkp1

In [ ]:
def update(A, B, C, D, xkp1_est, Pkp1, Q, R, yk, uk=None):
    """Carry out one complete time step of a Kalman filter
    (filtering form):
    1. Calculate the correction gain, Kf(k) and
        output estimation error covariance, S(k).
    2. Correct the estimates of the states and outputs at
        the current time using the current measurement, y(k):
            x_est(k|k-1) -> x_est(k|k)
            y_est(k|k-1) -> y_est(k|k)
    3. Calculate the predicted states and state error
        covariance at the next time instant:
            x_est(k+1|k)
            P(k+1|k)
        given the current input, u(k), and the system model.

    To do steps 1 and 2 only use: kalman_update_f.
    To do step 3 only use: kalman_predict_f.

    Numerical arguments are converted to cas.DM. Symbolic
    arguments (cas.SX or cas.MX) are passed through, so this
    function can be used to build a cas.Function.

    Arguments:
    A, B, C, D : matrices
        Discrete-time system model matrices.
    xkp1_est : (n, 1) vector
        Prior estimate of the states, x_est(k|k-1).
    Pkp1 : (n, n) matrix
        Prior estimate of the state error covariance, P(k|k-1).
    Q : (n, n) matrix
        State error covariance.
    R : (ny, ny) matrix
        Measurement error covariance.
    yk : vector, size (ny, 1)
        System output measurements at current time k.
    uk : vector, size (nu, 1), optional
        System inputs at current time k. Only optional if
        nu = 0.

    Returns:
    xk_est : (n, 1) vector
        Posterior estimate of the states, x_est(k|k).
    Pk : (n, n) matrix
        Covariance of state estimation error, P(k|k).
    yk_est : (ny, 1) vector
        Posterior estimate of the system output, y_est(k|k).
    Sk : (ny, ny) matrix
        Covariance of output prediction error, S(k).
    xkp1_est : (n, 1) vector
        Predicted states at next time instant, x_est(k+1|k).
    Pkp1 : (n, n) matrix
        Predicted state error covariance at next time instant,
        P(k+1|k).
    yk_pred : (ny, 1) vector
        Prior estimate of the system output, y_est(k|k-1).
    Kf : (n, ny) matrix
        Correction gain matrix.
    """
    if uk is None:
        uk = cas.DM.zeros(0)
    elif not isinstance(uk, (cas.SX, cas.MX)):
        uk = cas.DM(uk)
    if not isinstance(yk, (cas.SX, cas.MX)):
        yk = cas.DM(yk)

    n = A.shape[0]  # number of states
    nu = B.shape[1]  # number of inputs
    ny = C.shape[0]  # number of outputs
    assert A.shape[1] == n, "A.shape"
    assert B.shape[0] == n, "B.shape"
    assert C.shape[1] == n, "C.shape"
    assert D.shape == (ny, nu), "D.shape"

    # Check size of arguments passed
    assert uk.shape == (nu, 1), "uk.shape"
    assert yk.shape == (ny, 1), "yk.shape"

    # Update estimates based on current measurement
    # Predicted output at the current time
    yk_pred = linear_model_output(C, D, xkp1_est, uk)

    # Update states
    xk_est, Pk, Kf, Sk = kalman_update_f(
        C, R, xkp1_est, Pkp1, yk_pred, yk
    )

    # Calculate system output estimate
    yk_est = linear_model_output(C, D, xk_est, uk)

    # Predictions based on current inputs
    xkp1_est, Pkp1 = kalman_predict_f(A, B, Q, xk_est, Pk, uk)

    return xk_est, Pk, yk_est, Sk, xkp1_est, Pkp1, yk_pred, Kf

## Unit tests

In [ ]:
# Test system used in the tests below: 2 states, 1 input, 2 outputs
A_test = np.array([[0.7, 1.0], [0.0, 1.0]])
B_test = np.array([[1.0], [0.0]])
C_test = np.array([[0.3, 0.0], [0.0, 1.0]])
D_test = np.zeros((2, 1))
Q_test = np.diag([0.01, 0.01])
R_test = np.diag([0.1, 0.1])


def np_kalman_update(A, B, C, D, xkp1_est, Pkp1, Q, R, yk, uk):
    """NumPy version of the original JAX code (1-D vectors),
    used as a reference to test the CasADi functions.
    """
    yk_pred = C @ xkp1_est + D @ uk
    Sk = C @ Pkp1 @ C.T + R
    Kf = np.linalg.lstsq(Sk.T, (Pkp1 @ C.T).T, rcond=-1)[0].T
    xk_est = xkp1_est + Kf @ (yk - yk_pred)
    z = np.eye(A.shape[0]) - Kf @ C
    Pk = z @ Pkp1 @ z.T + Kf @ R @ Kf.T
    Pk = (Pk + Pk.T) / 2
    yk_est = C @ xk_est + D @ uk
    xkp1_est = A @ xk_est + B @ uk
    Pkp1 = A @ Pk @ A.T + Q
    return xk_est, Pk, yk_est, Sk, xkp1_est, Pkp1, yk_pred, Kf

In [ ]:
def test_linear_model_functions():
    A, B, C, D = map(cas.DM, (A_test, B_test, C_test, D_test))
    xk = cas.DM([1.0, 2.0])
    uk = cas.DM([0.5])

    xkp1 = linear_model_state_transition(A, B, xk, uk)
    assert isinstance(xkp1, cas.DM)
    assert xkp1.shape == (2, 1)
    np.testing.assert_allclose(xkp1, [[0.7 + 2.0 + 0.5], [2.0]])

    yk = linear_model_output(C, D, xk, uk)
    assert yk.shape == (2, 1)
    np.testing.assert_allclose(yk, [[0.3], [2.0]])

    # Non-zero D
    D = cas.DM([[1.0], [-1.0]])
    yk = linear_model_output(C, D, xk, uk)
    np.testing.assert_allclose(yk, [[0.3 + 0.5], [2.0 - 0.5]])

    # No inputs (nu = 0)
    xkp1 = linear_model_state_transition(
        A, cas.DM.zeros(2, 0), xk, cas.DM.zeros(0)
    )
    np.testing.assert_allclose(xkp1, A_test @ [[1.0], [2.0]])

    # Symbolic arguments
    for T in (cas.SX, cas.MX):
        xs, us = T.sym("x", 2), T.sym("u", 1)
        f = cas.Function(
            "f", [xs, us], [linear_model_state_transition(A, B, xs, us)]
        )
        np.testing.assert_allclose(f(xk, uk), [[3.2], [2.0]])


test_linear_model_functions()
print("test_linear_model_functions passed")

In [ ]:
def test_kalman_update_f():

    # Scalar system with known solution:
    # S = 1 * 1 * 1 + 1 = 2, Kf = 1 / 2,
    # x(k|k) = 0 + 0.5 * (4 - 0) = 2,
    # P(k|k) = 0.5^2 * 1 + 0.5^2 * 1 = 0.5
    one = cas.DM(1.0)
    xk_est, Pk, Kf, Sk = kalman_update_f(
        one, one, cas.DM(0.0), one, cas.DM(0.0), cas.DM(4.0)
    )
    for value in (xk_est, Pk, Kf, Sk):
        assert isinstance(value, cas.DM)
        assert value.shape == (1, 1)
    np.testing.assert_allclose(Sk, 2.0)
    np.testing.assert_allclose(Kf, 0.5)
    np.testing.assert_allclose(xk_est, 2.0)
    np.testing.assert_allclose(Pk, 0.5)

    # 2-state system compared to the standard form of the equations
    C, R = cas.DM(C_test), cas.DM(R_test)
    xk_pred = cas.DM([1.0, -1.0])
    Pk_pred = cas.DM([[2.0, 0.5], [0.5, 1.0]])
    yk_pred = C @ xk_pred
    yk = cas.DM([0.5, 0.0])
    xk_est, Pk, Kf, Sk = kalman_update_f(
        C, R, xk_pred, Pk_pred, yk_pred, yk
    )
    P, Cn = np.array(Pk_pred), C_test
    S = Cn @ P @ Cn.T + R_test
    K = P @ Cn.T @ np.linalg.inv(S)
    assert Kf.shape == (2, 2)
    np.testing.assert_allclose(Sk, S)
    np.testing.assert_allclose(Kf, K)
    np.testing.assert_allclose(
        xk_est, np.array(xk_pred) + K @ np.array(yk - yk_pred)
    )
    np.testing.assert_allclose(Pk, P - K @ S @ K.T)
    np.testing.assert_allclose(Pk, Pk.T)

    # Symbolic arguments
    for T in (cas.SX, cas.MX):
        x, P = T.sym("x", 2), T.sym("P", 2, 2)
        y = T.sym("y", 2)
        f = cas.Function(
            "f", [x, P, y], kalman_update_f(C, R, x, P, C @ x, y)
        )
        for a, b in zip(f(xk_pred, Pk_pred, yk), (xk_est, Pk, Kf, Sk)):
            np.testing.assert_allclose(a, b)


test_kalman_update_f()
print("test_kalman_update_f passed")

In [ ]:
def test_kalman_predict_f():
    A, B, Q = map(cas.DM, (A_test, B_test, Q_test))
    xk_est = cas.DM([1.0, 2.0])
    Pk = cas.DM([[2.0, 0.5], [0.5, 1.0]])
    uk = cas.DM([0.5])

    xkp1_est, Pkp1 = kalman_predict_f(A, B, Q, xk_est, Pk, uk)
    assert xkp1_est.shape == (2, 1)
    assert Pkp1.shape == (2, 2)
    np.testing.assert_allclose(xkp1_est, [[3.2], [2.0]])
    np.testing.assert_allclose(
        Pkp1, A_test @ np.array(Pk) @ A_test.T + Q_test
    )

    # Symbolic arguments
    for T in (cas.SX, cas.MX):
        x, P, u = T.sym("x", 2), T.sym("P", 2, 2), T.sym("u", 1)
        f = cas.Function(
            "f", [x, P, u], kalman_predict_f(A, B, Q, x, P, u)
        )
        for a, b in zip(f(xk_est, Pk, uk), (xkp1_est, Pkp1)):
            np.testing.assert_allclose(a, b)


test_kalman_predict_f()
print("test_kalman_predict_f passed")

In [ ]:
def test_update():
    A, B, C, D, Q, R = map(
        cas.DM, (A_test, B_test, C_test, D_test, Q_test, R_test)
    )

    # Compare to NumPy reference over 50 time steps with random data
    rng = np.random.default_rng(0)
    xkp1_est, Pkp1 = cas.DM.zeros(2), cas.DM.eye(2)
    xkp1_np, Pkp1_np = np.zeros(2), np.eye(2)
    for k in range(50):
        yk, uk = rng.normal(size=2), rng.normal(size=1)
        results = update(A, B, C, D, xkp1_est, Pkp1, Q, R, yk, uk)
        results_np = np_kalman_update(
            A_test, B_test, C_test, D_test, xkp1_np, Pkp1_np,
            Q_test, R_test, yk, uk
        )
        for a, b in zip(results, results_np):
            assert isinstance(a, cas.DM)
            np.testing.assert_allclose(
                np.array(a).reshape(b.shape), b, atol=1e-12
            )
        xkp1_est, Pkp1 = results[4], results[5]
        xkp1_np, Pkp1_np = results_np[4], results_np[5]

    # After many steps the gain should converge to the steady-state
    # Kalman filter gain from the discrete algebraic Riccati equation
    P_ss = scipy.linalg.solve_discrete_are(
        A_test.T, C_test.T, Q_test, R_test
    )
    Kf_ss = P_ss @ C_test.T @ np.linalg.inv(
        C_test @ P_ss @ C_test.T + R_test
    )
    np.testing.assert_allclose(Pkp1, P_ss, atol=1e-8)
    np.testing.assert_allclose(results[7], Kf_ss, atol=1e-8)

    # Lists and floats are accepted as yk and uk
    results = update(A, B, C, D, xkp1_est, Pkp1, Q, R, [1.0, 2.0], 0.5)
    assert results[0].shape == (2, 1)

    # No inputs (nu = 0), uk omitted
    results = update(
        A, cas.DM.zeros(2, 0), C, cas.DM.zeros(2, 0),
        cas.DM.zeros(2), cas.DM.eye(2), Q, R, [1.0, 2.0]
    )
    results_np = np_kalman_update(
        A_test, np.zeros((2, 0)), C_test, np.zeros((2, 0)),
        np.zeros(2), np.eye(2), Q_test, R_test, np.array([1.0, 2.0]),
        np.zeros(0)
    )
    for a, b in zip(results, results_np):
        np.testing.assert_allclose(np.array(a).reshape(b.shape), b)

    # Wrong-sized measurement raises an error
    try:
        update(A, B, C, D, xkp1_est, Pkp1, Q, R, [1.0], 0.5)
    except AssertionError as e:
        assert str(e) == "yk.shape"
    else:
        raise AssertionError("expected AssertionError")

    # Symbolic arguments: build a cas.Function of one time step
    for T in (cas.SX, cas.MX):
        x, P = T.sym("x", 2), T.sym("P", 2, 2)
        y, u = T.sym("y", 2), T.sym("u", 1)
        f = cas.Function(
            "kf_update", [x, P, y, u],
            list(update(A, B, C, D, x, P, Q, R, y, u))
        )
        expected = update(A, B, C, D, xkp1_est, Pkp1, Q, R, yk, uk)
        for a, b in zip(f(xkp1_est, Pkp1, yk, uk), expected):
            np.testing.assert_allclose(a, b)


test_update()
print("test_update passed")